<a href="https://colab.research.google.com/github/DTAMBU/03MAIR_Algoritmos_de_Optimizacion_2023_2024/blob/main/Damian_Tamburi_AG3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Guided Activity 3 - Optimization Algorithms

Name: Damian Tamburi

https://colab.research.google.com/drive/1RVxueryluA3YxYXzBX-ee-Y3iGuQHnCG

https://github.com/DTAMBU/03MAIR_Algoritmos_de_Optimizacion_2023_2024/blob/main/Damian_Tamburi_AG3.ipynb

In [ ]:
!pip install requests    #Make http calls to web pages
!pip install tsplib95    #Module for TSP problem instances

In [ ]:
import urllib.request #Make http calls to web pages
import tsplib95       #Module for TSP problem instances
import math           #Mathematical functions module. Used for exp
import random         #To generate random values


#http://elib.zib.de/pub/mp-testdata/tsp/tsplib/
#Documentation :
  # http://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp95.pdf
  # https://tsplib95.readthedocs.io/en/stable/pages/usage.html
  # https://tsplib95.readthedocs.io/en/v0.6.1/modules.html
  # https://pypi.org/project/tsplib95/

#Download the data file (Distance matrix)
file = "swiss42.tsp" ;
urllib.request.urlretrieve("http://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/swiss42.tsp.gz", file + '.gz')
!gzip -d swiss42.tsp.gz     #Decompress the data file

#Coordinates 51-city problem (Christofides/Eilon)
#file = "eil51.tsp" ; urllib.request.urlretrieve("http://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/eil51.tsp.gz", file)

#Coordinates - 48 capitals of the US (Padberg/Rinaldi)
#file = "att48.tsp" ; urllib.request.urlretrieve("http://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/att48.tsp.gz", file)

gzip: swiss42.tsp already exists; do you wish to overwrite (y or n)? ^C


In [ ]:
#Data loading and creation of the problem object
###############################################################################
problem = tsplib95.load(file)

#Nodes
Nodes = list(problem.get_nodes())

#Edges
Edges = list(problem.get_edges())

In [ ]:
#Let's try some functions of the problem object

#Distance between nodes
problem.get_weight(0, 1)

15

#Basic functions

In [ ]:
#Basic functions
###############################################################################

#Generates a random solution starting at node 0
def create_solution(Nodes):
  solution = [Nodes[0]]
  for n in Nodes[1:]:
    solution = solution + [random.choice(list(set(Nodes) - set({Nodes[0]}) - set(solution)))]
  return solution

#Returns the distance between two nodes
def distance(a,b, problem):
  return problem.get_weight(a,b)

#Returns the total distance of a path/solution
def total_distance(solution, problem):
  total_distance = 0
  for i in range(len(solution)-1):
    total_distance += distance(solution[i] ,solution[i+1] ,  problem)
  return total_distance + distance(solution[len(solution)-1] ,solution[0], problem)

temp_sol = create_solution(Nodes)

total_distance(temp_sol, problem), temp_sol

(4869,
 [0,
  19,
  31,
  4,
  20,
  15,
  6,
  16,
  32,
  34,
  14,
  13,
  24,
  27,
  36,
  22,
  38,
  35,
  28,
  1,
  18,
  2,
  23,
  9,
  30,
  12,
  7,
  41,
  8,
  17,
  26,
  33,
  10,
  11,
  21,
  3,
  39,
  29,
  25,
  37,
  40,
  5])

#RANDOM SEARCH

In [ ]:
###############################################################################
# RANDOM SEARCH
###############################################################################

def random_search(problem, N):
  #N is the number of iterations
  Nodes = list(problem.get_nodes())

  best_solution = []
  #best_distance = 10e100                           #Initialize with a high value
  best_distance = float('inf')                      #Initialize with a high value

  for i in range(N):                                #Stopping criterion: repeat N times, but others could be included
    solution = create_solution(Nodes)               #Generate a random solution
    distance = total_distance(solution, problem)    #Compute the objective value (total distance)

    if distance < best_distance:                    #Compare with the best obtained so far
      best_solution = solution
      best_distance = distance


  print("Best solution:" , best_solution)
  print("Distance     :" , best_distance)
  return best_solution


#Random search with 10000 iterations
solution = random_search(problem, 10000)

Best solution: [0, 1, 7, 22, 26, 20, 17, 32, 13, 37, 19, 12, 4, 23, 15, 40, 21, 9, 8, 30, 16, 34, 33, 38, 39, 10, 24, 41, 11, 25, 29, 31, 14, 28, 5, 36, 35, 27, 3, 2, 6, 18]
Distance     : 3766


#LOCAL SEARCH

In [ ]:
###############################################################################
# LOCAL SEARCH
###############################################################################
def generate_neighbor(solution):
  #Neighbor solution generator: 2-opt (swap 2 nodes). If there are N nodes, (N-1)x(N-2)/2 solutions are generated
  #It can be modified to apply generators other than 2-opt
  #print(solution)
  best_solution = []
  best_distance = 10e100
  for i in range(1,len(solution)-1):          #Traverse all nodes in a double loop to evaluate all 2-opt swaps
    for j in range(i+1, len(solution)):

      #A new solution is generated by swapping the two nodes i,j:
      #  (we use the + operator, which concatenates lists in python) : e.g.: [1,2] + [3] = [1,2,3]
      neighbor = solution[:i] + [solution[j]] + solution[i+1:j] + [solution[i]] + solution[j+1:]

      #The new solution is evaluated ...
      neighbor_distance = total_distance(neighbor, problem)

      #... to keep it if it improves on the previous ones
      if neighbor_distance <= best_distance:
        best_distance = neighbor_distance
        best_solution = neighbor
  return best_solution


#solution = [1, 47, 13, 41, 40, 19, 42, 44, 37, 5, 22, 28, 3, 2, 29, 21, 50, 34, 30, 9, 16, 11, 38, 49, 10, 39, 33, 45, 15, 24, 43, 26, 31, 36, 35, 20, 8, 7, 23, 48, 27, 12, 17, 4, 18, 25, 14, 6, 51, 46, 32]
print("Initial Solution Distance:" , total_distance(solution, problem))


new_solution = generate_neighbor(solution)
print("Best Local Solution Distance:", total_distance(new_solution, problem))

Initial Solution Distance: 3766
Best Local Solution Distance: 3328


In [ ]:
#Local Search:
#  - Based on the 2-opt neighborhood operator (generate_neighbor function)
#  - No stopping criterion, it stops when no further improvement is possible.
def local_search(problem):
  best_solution = []

  #Generate an initial reference solution (random)
  reference_solution = create_solution(Nodes)
  best_distance = total_distance(reference_solution, problem)

  iteration=0             #A counter to know how many iterations we do
  while(1):
    iteration +=1         #Increment the counter
    #print('#',iteration)

    #Get the best neighbor ...
    neighbor = generate_neighbor(reference_solution)

    #... and evaluate it to see whether we improve on what we found so far
    neighbor_distance = total_distance(neighbor, problem)

    #If we do not improve we must stop. We have reached a local minimum (according to our 2-opt neighborhood operator)
    if neighbor_distance < best_distance:
      #best_solution = copy.deepcopy(neighbor)   #With deep copy. Copies in python are by reference
      best_solution = neighbor                   #Keep the best solution found
      best_distance = neighbor_distance

    else:
      print("At iteration ", iteration, ", the best solution found is:" , best_solution)
      print("Distance     :" , best_distance)
      return best_solution

    reference_solution = neighbor


sol = local_search(problem )

At iteration  31 , the best solution found is: [0, 1, 5, 7, 31, 32, 34, 33, 20, 35, 36, 17, 37, 15, 16, 14, 19, 13, 26, 8, 40, 24, 21, 39, 2, 4, 6, 3, 27, 28, 29, 30, 38, 22, 9, 23, 41, 10, 25, 11, 12, 18]
Distance     : 1658


Variable neighborhood search:

the variable "n" is increased if a better solution is not found and decreased if a better solution is found. The idea is that if we are finding good solutions, then we are in a good region of the search space and we can afford to explore a smaller neighborhood. On the other hand, if we are not finding good solutions, then we might be trapped in a local minimum and we need to explore a larger neighborhood to try to escape.

In [ ]:
def generate_neighbor_2(solution, n):
  best_solution = []
  best_distance = 10e100
  for i in range(1,len(solution)-n):
    for j in range(i+n, len(solution)):

      neighbor = solution[:i] + solution[j:j+n] + solution[i+n:j] + solution[i:i+n] + solution[j+n:]

      neighbor_distance = total_distance(neighbor, problem)

      if neighbor_distance <= best_distance:
        best_distance = neighbor_distance
        best_solution = neighbor
  return best_solution

def adaptive_local_search(problem, N):
    Nodes = list(problem.get_nodes())
    best_solution = []
    best_distance = float('inf')

    #Generate an initial reference solution (random)
    reference_solution = create_solution(Nodes)
    best_distance = total_distance(reference_solution, problem)

    # Initialize the neighborhood size
    n = 2

    iteration=0             #A counter to know how many iterations we do
    while(1):
        iteration +=1         #Increment the counter

        #Get the best neighbor ...
        neighbor = generate_neighbor_2(reference_solution, n)

        #... and evaluate it to see whether we improve on what we found so far
        neighbor_distance = total_distance(neighbor, problem)

        #If we do not improve we must stop. We have reached a local minimum (according to our 2-opt neighborhood operator)
        if neighbor_distance < best_distance:
            best_solution = neighbor                   #Keep the best solution found
            best_distance = neighbor_distance
            n = max(2, n - 1)  # If we find a better solution, decrease n (but never below 2)
        else:
            n = min(len(Nodes) - 1, n + 1)  # If we do not find a better solution, increase n (but never above the number of nodes - 1)

        reference_solution = neighbor

        # Stopping condition: if n reaches its maximum value, we stop
        if n == len(Nodes) - 1:
            print("At iteration ", iteration, ", the best solution found is:" , best_solution)
            print("Distance     :" , best_distance)
            return best_solution

In [ ]:
sol2 = adaptive_local_search(problem, 1000)

At iteration  79 , the best solution found is: [0, 27, 3, 4, 5, 26, 18, 11, 10, 41, 23, 21, 40, 24, 39, 22, 38, 34, 20, 33, 32, 30, 28, 2, 29, 8, 9, 25, 12, 13, 19, 6, 1, 7, 14, 16, 15, 37, 17, 36, 35, 31]
Distance     : 1584


#SIMULATED ANNEALING

In [ ]:
###############################################################################
# SIMULATED ANNEALING
###############################################################################

#Generator of 1 neighbor solution, 2-opt, 100% random (swap 2 nodes)
#Could be improved by choosing another way to select a neighbor.
def generate_random_neighbor(solution):

  #Two nodes are chosen at random
  i,j = sorted(random.sample( range(1,len(solution)) , 2))

  #Returns a new solution with the two randomly chosen nodes swapped
  return solution[:i] + [solution[j]] + solution[i+1:j] + [solution[i]] + solution[j+1:]


#Probability function to accept worse solutions
def probability(T,d):
  if random.random() <  math.exp( -1*d / T)  :
    return True
  else:
    return False

#Temperature decrease function
def lower_temperature(T):
  return T*0.99

In [ ]:
def simulated_annealing(problem, TEMPERATURE ):
  #problem = problem data
  #T = Temperature

  reference_solution = create_solution(Nodes)
  reference_distance = total_distance(reference_solution, problem)

  best_solution = []             #x* in the pseudocode
  best_distance = 10e100         #F* in the pseudocode


  N=0
  while TEMPERATURE > .0001:
    N+=1
    #Generate a neighbor solution
    neighbor =generate_random_neighbor(reference_solution)

    #Compute its value (distance)
    neighbor_distance = total_distance(neighbor, problem)

    #If it is the best solution of all, it is saved (always!!!)
    if neighbor_distance < best_distance:
        best_solution = neighbor
        best_distance = neighbor_distance

    #If the new neighbor is better, we switch to it
    #If it is worse, we switch according to a probability that depends on T and delta(reference_distance - neighbor_distance)
    if neighbor_distance < reference_distance or probability(TEMPERATURE, abs(reference_distance - neighbor_distance) ) :
      #reference_solution = copy.deepcopy(neighbor)
      reference_solution = neighbor
      reference_distance = neighbor_distance

    #Lower the temperature
    TEMPERATURE = lower_temperature(TEMPERATURE)

  print("The best solution found is " , end="")
  print(best_solution)
  print("with a total distance of " , end="")
  print(best_distance)
  return best_solution

sol  = simulated_annealing(problem, 10000000)

The best solution found is [0, 26, 5, 13, 7, 31, 1, 6, 16, 15, 14, 18, 12, 10, 28, 30, 32, 36, 35, 20, 33, 39, 24, 40, 21, 9, 23, 41, 8, 4, 3, 2, 27, 34, 17, 37, 19, 11, 25, 29, 22, 38]
with a total distance of 2241


SIMULATED ANNEALING with local search:

In [ ]:
###############################################################################
# SIMULATED ANNEALING
###############################################################################

#Generator of 1 neighbor solution, 2-opt, 100% random (swap 2 nodes)
#Could be improved by choosing another way to select a neighbor.
def generate_random_neighbor(solution):

  #Two nodes are chosen at random
  i,j = sorted(random.sample( range(1,len(solution)) , 2))

  #Returns a new solution with the two randomly chosen nodes swapped
  return solution[:i] + [solution[j]] + solution[i+1:j] + [solution[i]] + solution[j+1:]


#Probability function to accept worse solutions
def probability(T,d):
  if random.random() <  math.exp( -1*d / T)  :
    return True
  else:
    return False

#Temperature decrease function
def lower_temperature(T):
  return T*0.99

In [ ]:
def simulated_annealing(problem, TEMPERATURE ):
  reference_solution = create_solution(Nodes)
  reference_distance = total_distance(reference_solution, problem)

  best_solution = []
  best_distance = 10e100

  N=0
  while TEMPERATURE > .0001:
    N+=1
    #Generate a neighbor solution
    neighbor = generate_neighbor(reference_solution)  # Modified here

    #Compute its value (distance)
    neighbor_distance = total_distance(neighbor, problem)

    #If it is the best solution of all, it is saved (always!!!)
    if neighbor_distance < best_distance:
        best_solution = neighbor
        best_distance = neighbor_distance

    #If the new neighbor is better, we switch to it
    #If it is worse, we switch according to a probability that depends on T and delta(reference_distance - neighbor_distance)
    if neighbor_distance < reference_distance or probability(TEMPERATURE, abs(reference_distance - neighbor_distance) ) :
      reference_solution = neighbor
      reference_distance = neighbor_distance

    #Lower the temperature
    TEMPERATURE = lower_temperature(TEMPERATURE)

  print("The best solution found is " , end="")
  print(best_solution)
  print("with a total distance of " , end="")
  print(best_distance)
  return best_solution

sol  = simulated_annealing(problem, 10000000)

The best solution found is [0, 3, 1, 7, 37, 17, 36, 35, 31, 32, 30, 21, 24, 40, 41, 10, 25, 11, 12, 18, 26, 4, 6, 5, 13, 19, 16, 15, 14, 2, 27, 28, 38, 22, 39, 9, 23, 8, 29, 34, 33, 20]
with a total distance of 1720


In [ ]:
def simulated_annealing(problem, TEMPERATURE ):
  reference_solution = create_solution(Nodes)
  reference_distance = total_distance(reference_solution, problem)

  best_solution = []
  best_distance = 10e100

  N=0
  while TEMPERATURE > .0001:
    N+=1
    #Generate a neighbor solution
    # Initialize the neighborhood size
    n = 2
    neighbor = generate_neighbor_2(reference_solution,n)  # Modified here

    #Compute its value (distance)
    neighbor_distance = total_distance(neighbor, problem)

    #If it is the best solution of all, it is saved (always!!!)
    if neighbor_distance < best_distance:
        best_solution = neighbor
        best_distance = neighbor_distance

    #If the new neighbor is better, we switch to it
    #If it is worse, we switch according to a probability that depends on T and delta(reference_distance - neighbor_distance)
    if neighbor_distance < reference_distance or probability(TEMPERATURE, abs(reference_distance - neighbor_distance) ) :
      reference_solution = neighbor
      reference_distance = neighbor_distance

    #Lower the temperature
    TEMPERATURE = lower_temperature(TEMPERATURE)

  print("The best solution found is " , end="")
  print(best_solution)
  print("with a total distance of " , end="")
  print(best_distance)
  return best_solution

sol  = simulated_annealing(problem, 10000000)

The best solution found is [0, 3, 2, 30, 28, 32, 34, 35, 36, 37, 14, 5, 19, 13, 16, 15, 17, 31, 20, 33, 38, 22, 40, 24, 21, 39, 29, 9, 8, 41, 23, 25, 12, 11, 10, 27, 26, 18, 6, 4, 1, 7]
with a total distance of 1805
